# Road Damage Classification — M2 + M3 | Google Colab

This notebook continues M1 using **the same crops and country splits**. Run the cells in order. Project training has not been performed in advance; this notebook contains no completed experiment results.

- **M2:** Train SmallCNN from scratch.
- **M3:** Run two independent experiments with ImageNet-pretrained ResNet18: train only the classification head with the backbone and BatchNorm statistics frozen, then fine-tune all layers in a separate experiment.
- Select the best checkpoint using **validation macro F1**.
- All experiments use seed 42, 224 × 224 images and 15 epochs. This is a practical comparison of different models; it does not isolate the effect of pretraining alone.
- Reports include learning curves, per-class precision/recall/F1, confusion matrices, top-1/top-2 accuracy, ECE, reliability diagrams, NLL, Brier score and an accuracy confidence interval.
- Save outputs to Drive and track experiments in W&B. Final test evaluation is a separate step after the experiments are fixed.

**Before starting:** Select a GPU runtime. Place the following files in one Drive folder, or add shortcuts to them under MyDrive:

- annotations.csv
- classification_manifest.csv
- split_manifest.csv
- classification_crops.zip

The required ZIP contains **the cropped images**. The GitHub repository ZIP is not a substitute. The full original images are not required for classification training.


## 1. Set up the environment

Use the PyTorch and Torchvision pair supplied by Colab, and install the W&B version specified in the M1 requirements. The actual package versions are recorded for each experiment.

If Colab requests a runtime restart after installation, restart it and run the setup cells again.


In [ ]:
import os, sys, subprocess, importlib.metadata
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
try:
    installed_wandb = importlib.metadata.version("wandb")
except importlib.metadata.PackageNotFoundError:
    installed_wandb = None
if installed_wandb != "0.19.8":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wandb==0.19.8"])
import torch, torchvision, numpy, pandas, matplotlib
print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime, then reconnect. Full training has not started.")
print("GPU:", torch.cuda.get_device_name(0))


## 2. Mount Drive and select the data folder

The path below is an **example**. Set DATA_FOLDER to your actual folder path: find the folder in Colab's Files panel and select Copy path.

Results are saved under OUTPUT_FOLDER. If you change the configuration of an experiment that has already started, use a new output folder.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# @title File paths and W&B account
DATA_FOLDER = "/content/drive/MyDrive/road-damage-data" # @param {type:"string"}
OUTPUT_FOLDER = "/content/drive/MyDrive/road_damage_M2_M3_runs/baseline_v1" # @param {type:"string"}
WANDB_ENTITY = "" # @param {type:"string"}
WANDB_PROJECT = "rdd2022-road-damage" # @param {type:"string"}

from pathlib import Path
DATA_DIR, OUTPUT_DIR = Path(DATA_FOLDER), Path(OUTPUT_FOLDER)
required_files = ["annotations.csv", "classification_manifest.csv", "split_manifest.csv", "classification_crops.zip"]
missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Check DATA_FOLDER. Missing files: {missing}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data:", DATA_DIR)
print("Results:", OUTPUT_DIR)


## 3. Prepare the M2 and M3 code

The training code is embedded in this notebook, so separate Python file uploads are not required. A copy of the code is saved with each experiment for review and reproduction.


In [ ]:
MODULE_SOURCES = {'m2m3_data.py': '"""Read-only checks of the fixed M1 manifests and safe crop extraction."""\nfrom pathlib import Path, PurePosixPath\nimport hashlib\nimport json\nimport shutil\nimport stat\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nCLASSES = ["D00", "D10", "D20", "D40"]\nSPLITS = ["train", "val", "test"]\nFILES = ["annotations.csv", "classification_manifest.csv", "split_manifest.csv"]\nEXPECTED_COUNTS = {"train": 12000, "val": 9463, "test": 11229}\nEXPECTED_SOURCES = {"India": "train", "Japan": "train", "United_States": "train",\n                    "China": "val", "Czech": "val", "Norway": "test"}\n\n\ndef require(ok, message):\n    if not bool(ok):\n        raise ValueError(message)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef save_json(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef crop_relative(value):\n    """Resolve Windows separators without using absolute raw-data paths."""\n    value = str(value).replace("\\\\", "/")\n    parts = PurePosixPath(value).parts\n    require(not value.startswith("/") and not any(":" in p for p in parts), "Absolute crop path rejected")\n    require(".." not in parts and parts.count("classification_crops") == 1, "Unsafe or unrecognized crop path")\n    tail = parts[parts.index("classification_crops") + 1:]\n    require(len(tail) == 3 and tail[0] in SPLITS and tail[1] in CLASSES, "Expected split/class/file.jpg")\n    require(tail[2].lower().endswith(".jpg"), "Expected JPG crop")\n    return "/".join(tail)\n\n\ndef validate_manifests(data_dir, strict=True):\n    """Validate tables, not pixels or visual correctness of damage labels."""\n    data_dir = Path(data_dir)\n    a = pd.read_csv(data_dir / FILES[0], keep_default_na=False)\n    c = pd.read_csv(data_dir / FILES[1], keep_default_na=False)\n    s = pd.read_csv(data_dir / FILES[2], dtype={"phash": str, "sha256": str}, keep_default_na=False)\n    required = [\n        (a, ["image_id", "object_index", "class", "country", "source", "image_path", "split", "xmin", "ymin", "xmax", "ymax"]),\n        (c, ["image_id", "crop_path", "class", "split"]),\n        (s, ["image_id", "country", "source", "image_path", "width", "height", "sha256", "split", "status"]),\n    ]\n    for frame, cols in required:\n        require(set(cols) <= set(frame.columns), f"Missing columns: {set(cols) - set(frame.columns)}")\n        require(len(frame) > 0 and not frame[cols].isna().any().any(), "Missing required values")\n        require(not frame[cols].astype(str).apply(lambda col: col.str.strip().eq("")).any().any(), "Blank required values")\n        require(frame["split"].isin(SPLITS).all(), "Unknown split")\n    require(not s.image_id.duplicated().any(), "Duplicate parent image ID")\n    require(s.status.eq("valid").all(), "Invalid parent images")\n    require((s[["width", "height"]] > 0).all().all(), "Invalid parent dimensions")\n    require(s.sha256.str.fullmatch(r"[a-fA-F0-9]{64}").all(), "Invalid recorded SHA256")\n    require(not s.sha256.duplicated().any(), "Recorded exact duplicate images remain")\n    require((s.groupby("source").split.nunique() == 1).all(), "Country/source crosses splits")\n    require(a["class"].isin(CLASSES).all() and c["class"].isin(CLASSES).all(), "Unknown class")\n    require(not a.duplicated(["image_id", "object_index"]).any(), "Duplicate annotation object key")\n    indices = pd.to_numeric(a.object_index, errors="coerce")\n    require(np.isfinite(indices).all() and indices.ge(0).all() and indices.eq(indices.round()).all(), "Invalid object index")\n    require(not c.crop_path.duplicated().any(), "Duplicate crop path")\n    parent = a.merge(s, on="image_id", how="left", suffixes=("", "_parent"), validate="many_to_one", indicator=True)\n    require(parent._merge.eq("both").all(), "Annotation parent missing")\n    for col in ["split", "country", "source", "image_path"]:\n        require(parent[col].eq(parent[col + "_parent"]).all(), f"Annotation {col} differs from parent")\n    boxes = parent[["xmin", "ymin", "xmax", "ymax"]].to_numpy(dtype=float)\n    require(np.isfinite(boxes).all(), "Non-finite bounding box")\n    require(((boxes[:, 0] >= 0) & (boxes[:, 1] >= 0) & (boxes[:, 2] > boxes[:, 0]) &\n             (boxes[:, 3] > boxes[:, 1]) & (boxes[:, 2] <= parent.width) &\n             (boxes[:, 3] <= parent.height)).all(), "Invalid or out-of-bounds bounding box")\n    c = c.copy()\n    c["relative_path"] = c.crop_path.map(crop_relative)\n    require(not c.relative_path.duplicated().any(), "Duplicate normalized crop path")\n    tails = c.relative_path.str.split("/")\n    require(tails.str[0].eq(c.split).all() and tails.str[1].eq(c["class"]).all(), "Crop folder label/split mismatch")\n    parsed = tails.str[2].str.extract(r"^(.*)_(\\d+)\\.jpg$")\n    require(parsed.notna().all().all() and parsed[0].eq(c.image_id).all(), "Crop filename does not match parent")\n    c["object_index"] = parsed[1].astype(int)\n    require(not c.duplicated(["image_id", "object_index"]).any(), "Duplicate crop object key")\n    c = c.merge(a, on=["image_id", "object_index"], how="left", suffixes=("", "_annotation"), validate="one_to_one", indicator=True)\n    require(c._merge.eq("both").all(), "Crop has no source annotation")\n    for col in ["class", "split"]:\n        require(c[col].eq(c[col + "_annotation"]).all(), f"Crop {col} does not match annotation")\n    require((c.groupby("image_id").split.nunique() == 1).all(), "Parent image leaks across splits")\n    require(not ((c.image_id == "Japan_Japan_003187") & (c.object_index == 0)).any(), "Excluded Japan crop is present")\n    for split in ["val", "test"]:\n        required_keys = set(a.loc[a.split.eq(split), ["image_id", "object_index"]].itertuples(index=False, name=None))\n        actual_keys = set(c.loc[c.split.eq(split), ["image_id", "object_index"]].itertuples(index=False, name=None))\n        require(required_keys == actual_keys, f"{split} crops do not exactly cover source annotations")\n    counts = c.split.value_counts().to_dict()\n    if strict:\n        require(len(a) == 55002 and len(s) == 38384 and counts == EXPECTED_COUNTS, "M1 snapshot counts changed; review before training")\n        require(s.groupby("source").split.first().to_dict() == EXPECTED_SOURCES, "M1 country allocation changed")\n        require(c.loc[c.split.eq("train")].groupby("class").size().to_dict() == dict.fromkeys(CLASSES, 3000), "M1 balanced training subset changed")\n    balance = pd.crosstab(c["class"], c.split).reindex(index=CLASSES, columns=SPLITS, fill_value=0)\n    report = {\n        "status": "PASS", "scope": "Manifest consistency only; pixels and visual labels not checked here.",\n        "original_images": len(s), "annotations": len(a), "crops": len(c),\n        "crop_counts": {k: int(v) for k, v in counts.items()},\n        "class_counts": {k: {sp: int(balance.loc[k, sp]) for sp in SPLITS} for k in CLASSES},\n        "parent_counts": {k: int(v) for k, v in c.groupby("split").image_id.nunique().items()},\n        "country_allocation": s.groupby("source").split.first().to_dict(),\n        "excluded_japan_annotation_present": bool(((a.image_id == "Japan_Japan_003187") & (a.object_index == 0)).any()),\n        "excluded_japan_crop_present": False,\n        "recorded_sha256_duplicates": int(s.sha256.duplicated().sum()),\n        "manifest_sha256": {name: sha256(data_dir / name) for name in FILES},\n    }\n    c = c.drop(columns=["_merge", "class_annotation", "split_annotation"])\n    return c, report\n\n\ndef extract_crops(archive, destination):\n    """Extract known-layout JPGs to a fresh local directory without overwriting."""\n    archive, destination = Path(archive), Path(destination)\n    digest = sha256(archive)\n    marker = destination / ".archive.json"\n    if marker.exists():\n        require(json.loads(marker.read_text())["sha256"] == digest, "Crop cache belongs to another ZIP")\n        return destination\n    require(not destination.exists() or not any(destination.iterdir()),\n            "Nonempty crop cache without completion marker. Choose a new local cache directory.")\n    with zipfile.ZipFile(archive) as z:\n        entries, targets = [], set()\n        for info in z.infolist():\n            parts = PurePosixPath(info.filename.replace("\\\\", "/")).parts\n            require(".." not in parts and not info.filename.startswith(("/", "\\\\")) and not any(":" in p for p in parts), "Unsafe ZIP member")\n            require(not stat.S_ISLNK(info.external_attr >> 16), "ZIP symlinks rejected")\n            if info.is_dir() or "__MACOSX" in parts or not info.filename.lower().endswith(".jpg"):\n                continue\n            tail = parts[parts.index("classification_crops") + 1:] if "classification_crops" in parts else parts\n            require(len(tail) == 3 and tail[0] in SPLITS and tail[1] in CLASSES, f"Unexpected ZIP layout: {info.filename}")\n            relative = "/".join(tail)\n            require(relative not in targets, "Duplicate ZIP image name")\n            targets.add(relative)\n            entries.append((info, relative))\n        require(bool(entries), "No crop JPGs found")\n        require(sum(info.file_size for info, _ in entries) < 10 * 1024**3, "Unexpectedly large archive")\n        destination.mkdir(parents=True, exist_ok=True)\n        for info, relative in entries:\n            target = destination / relative\n            target.parent.mkdir(parents=True, exist_ok=True)\n            with z.open(info) as src, target.open("xb") as dst:\n                shutil.copyfileobj(src, dst)\n    save_json(marker, {"sha256": digest, "files": len(entries)})\n    return destination\n\n\ndef verify_pixels(crops, crop_root, image_size=224):\n    """Decode every image once and verify RGB, size and manifest/file agreement."""\n    crop_root = Path(crop_root)\n    expected = set(crops.relative_path)\n    actual = {p.relative_to(crop_root).as_posix() for p in crop_root.rglob("*") if p.is_file() and p.suffix.lower() == ".jpg"}\n    require(actual == expected, f"Crop files differ: {len(expected-actual)} missing, {len(actual-expected)} extra")\n    for index, rel in enumerate(crops.relative_path, 1):\n        with Image.open(crop_root / rel) as im:\n            im.load()\n            require(im.mode == "RGB" and im.size == (image_size, image_size), f"Unexpected crop mode/size: {rel}")\n        if index % 5000 == 0:\n            print(f"Verified {index:,}/{len(crops):,} images", flush=True)\n    return {"status": "PASS", "images_decoded": len(crops), "mode": "RGB", "size": [image_size, image_size]}\n\n', 'm2m3_metrics.py': '"""Classification metrics, calibration, error evidence and a provisional improvement plan."""\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom PIL import Image\nfrom m2m3_data import CLASSES, require, save_json\n\n\ndef classification_metrics(labels, probabilities, bins=10):\n    y = np.asarray(labels, dtype=int)\n    p = np.asarray(probabilities, dtype=float)\n    require(len(y) > 0 and p.shape == (len(y), len(CLASSES)), "Invalid prediction shape")\n    require(np.isfinite(p).all() and (p >= 0).all() and (p <= 1).all(), "Invalid probabilities")\n    require(np.allclose(p.sum(axis=1), 1, atol=1e-5), "Probabilities must sum to one")\n    require(((y >= 0) & (y < len(CLASSES))).all(), "Invalid label IDs")\n    pred = p.argmax(axis=1)\n    matrix = np.bincount(y * len(CLASSES) + pred, minlength=len(CLASSES)**2).reshape(len(CLASSES), len(CLASSES))\n    support, predicted = matrix.sum(axis=1), matrix.sum(axis=0)\n    precision = np.divide(matrix.diagonal(), predicted, out=np.zeros(4), where=predicted > 0)\n    recall = np.divide(matrix.diagonal(), support, out=np.zeros(4), where=support > 0)\n    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(4), where=(precision + recall) > 0)\n    confidence, correct = p.max(axis=1), pred == y\n    bin_ids = np.minimum((confidence * bins).astype(int), bins - 1)\n    calibration, ece = [], 0.0\n    for index in range(bins):\n        mask = bin_ids == index\n        count = int(mask.sum())\n        acc = float(correct[mask].mean()) if count else None\n        conf = float(confidence[mask].mean()) if count else None\n        if count:\n            ece += count / len(y) * abs(acc - conf)\n        calibration.append({"bin": index, "lower": index / bins, "upper": (index + 1) / bins,\n                            "count": count, "accuracy": acc, "confidence": conf})\n    metrics = {\n        "accuracy": float(correct.mean()),\n        "top2_accuracy": float((np.argsort(-p, axis=1, kind="stable")[:, :2] == y[:, None]).any(axis=1).mean()),\n        "macro_precision": float(precision.mean()), "macro_recall": float(recall.mean()), "macro_f1": float(f1.mean()),\n        "ece": float(ece), "nll": float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean()),\n        "brier": float(((p - np.eye(4)[y]) ** 2).sum(axis=1).mean()),\n        "per_class": {name: {"precision": float(precision[i]), "recall": float(recall[i]),\n                             "f1": float(f1[i]), "support": int(support[i])} for i, name in enumerate(CLASSES)},\n        "calibration_bins": calibration,\n    }\n    return metrics, matrix\n\n\ndef parent_bootstrap_accuracy(labels, probabilities, parent_ids, draws=2000, seed=42):\n    """Percentile CI resampling parent images; all their crops travel together."""\n    correct = np.asarray(probabilities).argmax(axis=1) == np.asarray(labels)\n    _, group = np.unique(np.asarray(parent_ids), return_inverse=True)\n    n = group.max() + 1\n    sizes = np.bincount(group)\n    successes = np.bincount(group, weights=correct.astype(int))\n    rng = np.random.default_rng(seed)\n    estimates = np.empty(draws)\n    for i in range(draws):\n        chosen = rng.integers(0, n, size=n)\n        estimates[i] = successes[chosen].sum() / sizes[chosen].sum()\n    lo, hi = np.quantile(estimates, [0.025, 0.975])\n    return {"method": "parent-image cluster bootstrap, percentile 95%",\n            "lower": float(lo), "upper": float(hi), "draws": draws, "parent_images": int(n),\n            "limitation": "Conditional on this held-out country and trained model; not a multi-seed or cross-country CI."}\n\n\ndef prediction_frame(rows, labels, probabilities):\n    out = rows[["relative_path", "image_id", "object_index", "country", "source", "split", "class"]].reset_index(drop=True).copy()\n    require(np.array_equal(np.asarray(labels), out["class"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()), "Prediction order mismatch")\n    pred = probabilities.argmax(axis=1)\n    out["predicted"] = [CLASSES[i] for i in pred]\n    out["confidence"] = probabilities.max(axis=1)\n    out["correct"] = out["class"].eq(out.predicted)\n    for i, name in enumerate(CLASSES):\n        out[f"prob_{name}"] = probabilities[:, i]\n    return out\n\n\ndef plot_confusion(matrix, path, split):\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))\n    denominators = matrix.sum(axis=1, keepdims=True)\n    rates = np.divide(matrix, denominators, out=np.zeros_like(matrix, dtype=float), where=denominators > 0)\n    for ax, values, title, fmt in zip(axes, [matrix, rates], ["Counts", "Recall by true class"], ["d", ".2f"]):\n        im = ax.imshow(values, cmap="Blues", vmin=0)\n        fig.colorbar(im, ax=ax, shrink=0.8)\n        ax.set(xticks=range(4), yticks=range(4), xticklabels=CLASSES, yticklabels=CLASSES,\n               xlabel="Predicted", ylabel="True", title=f"{split}: {title}")\n        for i in range(4):\n            for j in range(4):\n                ax.text(j, i, format(values[i, j], fmt), ha="center", va="center",\n                        color="white" if values[i, j] > values.max() / 2 else "black")\n    fig.tight_layout()\n    fig.savefig(path, dpi=160)\n    plt.close(fig)\n\n\ndef plot_calibration(metrics, path, split):\n    bins = pd.DataFrame(metrics["calibration_bins"])\n    populated = bins[bins["count"] > 0]\n    fig, axes = plt.subplots(1, 2, figsize=(10, 4))\n    axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")\n    axes[0].plot(populated.confidence, populated.accuracy, "o-", label="Raw softmax")\n    axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean confidence", ylabel="Accuracy",\n                title=f"{split}: ECE = {metrics[\'ece\']:.4f}")\n    axes[0].legend()\n    axes[1].bar((bins.lower + bins.upper) / 2, bins["count"], width=0.09)\n    axes[1].set(xlabel="Confidence", ylabel="Number of crops", title="Confidence distribution")\n    fig.tight_layout()\n    fig.savefig(path, dpi=160)\n    plt.close(fig)\n\n\ndef plot_errors(frame, crop_root, path, split):\n    wrong = frame[~frame.correct].sort_values("confidence", ascending=False).head(16)\n    fig, axes = plt.subplots(4, 4, figsize=(11, 11))\n    for ax in axes.ravel():\n        ax.axis("off")\n    if wrong.empty:\n        fig.text(0.5, 0.5, "No errors in this evaluated set", ha="center")\n    for ax, (_, row) in zip(axes.ravel(), wrong.iterrows()):\n        with Image.open(Path(crop_root) / row.relative_path) as im:\n            ax.imshow(im)\n        ax.set_title(f"{row[\'class\']} -> {row.predicted} ({row.confidence:.2f})\\n{Path(row.relative_path).stem[-27:]}", fontsize=8)\n    fig.suptitle(f"{split}: highest-confidence errors (not a random sample)")\n    fig.tight_layout()\n    fig.savefig(path, dpi=150)\n    plt.close(fig)\n\n\ndef plot_history(history, path):\n    frame = pd.DataFrame(history)\n    fig, axes = plt.subplots(1, 3, figsize=(14, 4))\n    for ax, metric, title in zip(axes, ["loss", "accuracy", "macro_f1"], ["Loss", "Accuracy", "Macro F1"]):\n        for split in ["train", "val"]:\n            key = f"{split}/{metric}"\n            if key in frame:\n                ax.plot(frame.epoch, frame[key], label=split)\n        ax.set(xlabel="Epoch", title=title)\n        ax.legend()\n    fig.tight_layout()\n    fig.savefig(path, dpi=150)\n    plt.close(fig)\n\n\ndef improvement_plan(frame, matrix, path):\n    """Quantified validation evidence, with explicit human-review tasks, not invented causes."""\n    lines = [\n        "# Validation-based data-improvement plan",\n        "",\n        "Status: provisional until the error-review CSV is completed by a person.",\n        "Counts below come from validation predictions. They do not establish why a label was misclassified.",\n        "Use training data or newly collected training examples for improvements. Keep validation and Norway test fixed.",\n        "",\n    ]\n    priority = []\n    for i, name in enumerate(CLASSES):\n        support = int(matrix[i].sum())\n        misses = support - int(matrix[i, i])\n        recall = matrix[i, i] / support if support else 0\n        priority.append((recall, name, support, misses))\n    for rank, (recall, name, support, misses) in enumerate(sorted(priority), 1):\n        lines.append(f"{rank}. {name}: validation recall {recall:.3f}, {misses}/{support} missed. "\n                     "Review original-resolution evidence and matched training examples; if coverage is deficient, "\n                     "add verified training examples and compare validation macro F1 and this class\'s recall.")\n    pairs = [(int(matrix[i, j]), CLASSES[i], CLASSES[j]) for i in range(4) for j in range(4) if i != j and matrix[i, j]]\n    lines += ["", "## Largest observed confusion pairs", ""]\n    for count, truth, pred in sorted(pairs, reverse=True)[:5]:\n        lines.append(f"- {truth} -> {pred}: {count} errors. Review both classes for ambiguous labels, narrow crops, lighting and road texture; record the observed cause before selecting a fix.")\n    high = int(((~frame.correct) & (frame.confidence >= 0.9)).sum())\n    lines += [\n        "", f"High-confidence errors (confidence >= 0.90): {high}. Inspect these first, then a random error sample.",\n        "", "## Review and follow-up", "",\n        "- Complete error_review.csv: observed_cause, evidence_note, proposed_action.",\n        "- Count reviewed causes and replace these hypotheses with actions supported by those counts.",\n        "- For confirmed training-label errors: record image_id/object_index, obtain team agreement, and version corrected labels.",\n        "- For crop/context problems: first inspect original images; any recropping experiment must use the same parent splits and a separately versioned dataset.",\n        "- For domain coverage problems: collect licensed training images of the underperforming damage classes under the observed conditions.",\n        "- Re-evaluate proposed fixes on validation before the final test. Test errors may inform future work, but do not retune and present the same test as untouched.",\n        "",\n        "Known M1 audit caveat: row-number-based decisions appear to refer to different image IDs across two audit samples. Resolve by image_id/object_index before claiming a label-error rate or relabeling.",\n    ]\n    Path(path).write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n\n\ndef save_evaluation(rows, labels, probabilities, crop_root, output, split, seed=42, bootstrap_draws=2000):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    metrics, matrix = classification_metrics(labels, probabilities)\n    metrics["accuracy_ci95"] = parent_bootstrap_accuracy(labels, probabilities, rows.image_id, bootstrap_draws, seed)\n    metrics["calibration_definition"] = "Uncalibrated softmax; top-label ECE with 10 equal-width bins; multiclass Brier=sum squared class errors."\n    frame = prediction_frame(rows, labels, probabilities)\n    frame.to_csv(output / "predictions.csv", index=False)\n    per_class = pd.DataFrame(metrics["per_class"]).T.rename_axis("class").reset_index()\n    per_class.to_csv(output / "per_class_metrics.csv", index=False)\n    pd.DataFrame(metrics["calibration_bins"]).to_csv(output / "calibration_bins.csv", index=False)\n    pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).to_csv(output / "confusion_matrix.csv", index_label="true_class")\n    countries = []\n    for country, part in frame.groupby("country", sort=True):\n        y = part["class"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()\n        p = part[[f"prob_{c}" for c in CLASSES]].to_numpy()\n        m, _ = classification_metrics(y, p)\n        countries.append({"country": country, "crops": len(part), **{k: v for k, v in m.items() if isinstance(v, (int, float))}})\n    pd.DataFrame(countries).to_csv(output / "country_metrics.csv", index=False)\n    plot_confusion(matrix, output / "confusion_matrix.png", split)\n    plot_calibration(metrics, output / "reliability.png", split)\n    plot_errors(frame, crop_root, output / "error_gallery.png", split)\n    wrong = frame[~frame.correct]\n    top = wrong.sort_values("confidence", ascending=False).head(16).copy()\n    top["review_selection"] = "highest_confidence"\n    remaining = wrong.drop(index=top.index)\n    random_errors = remaining.sample(n=min(32, len(remaining)), random_state=seed).copy()\n    random_errors["review_selection"] = "random_remaining_errors"\n    review = pd.concat([top, random_errors], ignore_index=True)\n    for col in ["observed_cause", "evidence_note", "proposed_action"]:\n        review[col] = ""\n    # Never overwrite a human\'s completed review.\n    if not (output / "error_review.csv").exists():\n        review.to_csv(output / "error_review.csv", index=False)\n    if split == "val":\n        improvement_plan(frame, matrix, output / "data_improvement_plan.md")\n    save_json(output / "metrics.json", metrics)\n    return metrics\n\n\ndef flat_metrics(metrics, prefix):\n    out = {f"{prefix}/{key}": value for key, value in metrics.items() if isinstance(value, (int, float))}\n    for name, values in metrics["per_class"].items():\n        out.update({f"{prefix}/{name}/{key}": value for key, value in values.items()})\n    if "accuracy_ci95" in metrics:\n        out[f"{prefix}/accuracy_ci95_lower"] = metrics["accuracy_ci95"]["lower"]\n        out[f"{prefix}/accuracy_ci95_upper"] = metrics["accuracy_ci95"]["upper"]\n    return out\n\n', 'm2m3_models.py': '"""Model definitions: retain M1 SmallCNN; freeze ResNet buffers as well as weights."""\nimport random\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torchvision import models, transforms\n\nSTAGES = ["scratch", "frozen", "finetune"]\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n    torch.use_deterministic_algorithms(True, warn_only=True)\n\n\ndef seed_worker(worker_id):\n    seed = torch.initial_seed() % 2**32\n    np.random.seed(seed)\n    random.seed(seed)\n\n\nclass SmallCNN(nn.Module):\n    def __init__(self, classes=4):\n        super().__init__()\n        self.features = nn.Sequential(\n            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),\n            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),\n            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(1))\n        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, classes))\n\n    def forward(self, value):\n        return self.classifier(self.features(value))\n\n\ndef build_model(stage, pretrained=True):\n    if stage not in STAGES:\n        raise ValueError(f"Unknown stage: {stage}")\n    if stage == "scratch":\n        return SmallCNN()\n    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None\n    model = models.resnet18(weights=weights)\n    model.fc = nn.Linear(model.fc.in_features, 4)\n    if stage == "frozen":\n        for p in model.parameters():\n            p.requires_grad = False\n        for p in model.fc.parameters():\n            p.requires_grad = True\n    return model\n\n\ndef training_mode(model, stage):\n    if stage == "frozen":\n        model.eval()  # Keep BatchNorm running statistics fixed.\n        model.fc.train()\n    else:\n        model.train()\n\n\ndef image_transform(training, size=224):\n    steps = [transforms.Resize((size, size))]\n    if training:\n        # Mild transforms retain the precomputed crop extent; no RandomResizedCrop.\n        steps += [transforms.RandomHorizontalFlip(),\n                  transforms.RandomRotation(10, interpolation=transforms.InterpolationMode.BILINEAR,\n                                            fill=(124, 116, 104))]\n    steps += [transforms.ToTensor(),\n              transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]\n    return transforms.Compose(steps)\n\n\ndef atomic_checkpoint(path, payload):\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(payload, temporary)\n    temporary.replace(path)\n\n', 'm2m3_train.py': '"""Colab training and final evaluation. No test inference is performed by train_experiment."""\nimport copy\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport uuid\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom m2m3_data import CLASSES, require, save_json, sha256\nfrom m2m3_models import STAGES, build_model, training_mode, seed_everything, seed_worker, image_transform, atomic_checkpoint\nfrom m2m3_metrics import classification_metrics, flat_metrics, plot_history, save_evaluation\n\nDEFAULT_CONFIG = {\n    "seed": 42, "image_size": 224, "batch_size": 32, "num_workers": 2, "epochs": 15,\n    "scratch_lr": 0.001, "transfer_lr": 0.0003, "weight_decay": 0.0001,\n    "wandb_project": "rdd2022-road-damage", "wandb_entity": None, "wandb_mode": "online",\n    "group": "m2m3-baseline-seed42", "bootstrap_draws": 2000,\n    "augmentation": "resize224; horizontal_flip_p0.5; rotation_10deg; ImageNet_normalize",\n    "checkpoint_selection": "maximum validation macro F1; earliest epoch on ties",\n    "pretrained_weights": "ResNet18_Weights.IMAGENET1K_V1",\n}\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()\n\n\ndef source_hashes():\n    root = Path(__file__).parent\n    return {p.name: sha256(p) for p in sorted(root.glob("m2m3_*.py"))}\n\n\ndef environment():\n    versions = {}\n    for name in ["torch", "torchvision", "numpy", "pandas", "Pillow", "matplotlib", "wandb"]:\n        try:\n            versions[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            versions[name] = "not-installed"\n    return {"python": sys.version, "platform": platform.platform(), "packages": versions,\n            "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}\n\n\nclass CropDataset(Dataset):\n    def __init__(self, rows, crop_root, training, size=224):\n        self.rows = rows.reset_index(drop=True)\n        self.paths = [Path(crop_root) / p for p in self.rows.relative_path]\n        self.labels = self.rows["class"].map({name: i for i, name in enumerate(CLASSES)}).to_numpy()\n        self.transform = image_transform(training, size)\n\n    def __len__(self):\n        return len(self.paths)\n\n    def __getitem__(self, index):\n        with Image.open(self.paths[index]) as image:\n            tensor = self.transform(image.convert("RGB"))\n        return tensor, int(self.labels[index])\n\n\ndef make_loader(rows, crop_root, cfg, training=False, generator=None):\n    return DataLoader(CropDataset(rows, crop_root, training, cfg["image_size"]),\n                      batch_size=cfg["batch_size"], shuffle=training,\n                      num_workers=cfg["num_workers"], pin_memory=torch.cuda.is_available(),\n                      worker_init_fn=seed_worker, generator=generator, persistent_workers=False)\n\n\ndef evaluate(model, loader, device):\n    model.eval()\n    labels, probabilities, loss_sum = [], [], 0.0\n    with torch.inference_mode():\n        for x, y in loader:\n            x, y = x.to(device), y.to(device)\n            logits = model(x)\n            loss_sum += nn.functional.cross_entropy(logits, y, reduction="sum").item()\n            labels.append(y.cpu().numpy())\n            probabilities.append(logits.softmax(dim=1).cpu().numpy())\n    return np.concatenate(labels), np.concatenate(probabilities), loss_sum / len(loader.dataset)\n\n\nclass Tracker:\n    """Disable only for local smoke tests. Final submission requires online W&B runs."""\n    def __init__(self, cfg, stage, directory, kind="train", run_id=None):\n        self.run = None\n        self.wandb = None\n        if cfg["wandb_mode"] != "disabled":\n            import wandb\n            self.wandb = wandb\n            self.run = wandb.init(project=cfg["wandb_project"], entity=cfg["wandb_entity"],\n                                  mode=cfg["wandb_mode"], group=cfg["group"], job_type=kind,\n                                  name=f"{stage}-{kind}-{cfg[\'seed\']}", config={**cfg, "stage": stage},\n                                  id=run_id, resume="allow" if run_id and cfg["wandb_mode"] == "online" else None,\n                                  dir=str(directory))\n            if kind == "train":\n                self.run.define_metric("epoch")\n                self.run.define_metric("train/*", step_metric="epoch")\n                self.run.define_metric("val/*", step_metric="epoch")\n\n    def log(self, values):\n        if self.run:\n            self.run.log(values)\n\n    def summary(self, values):\n        if self.run:\n            self.run.summary.update(values)\n\n    def reports(self, directory, prefix):\n        if not self.run:\n            return\n        directory = Path(directory)\n        self.log({f"{prefix}/confusion_matrix": self.wandb.Image(str(directory / "confusion_matrix.png")),\n                  f"{prefix}/reliability": self.wandb.Image(str(directory / "reliability.png")),\n                  f"{prefix}/error_gallery": self.wandb.Image(str(directory / "error_gallery.png")),\n                  f"{prefix}/per_class": self.wandb.Table(dataframe=pd.read_csv(directory / "per_class_metrics.csv")),\n                  f"{prefix}/by_country": self.wandb.Table(dataframe=pd.read_csv(directory / "country_metrics.csv"))})\n        # Metrics, prediction probabilities and review tables only; no source-image dataset upload.\n        artifact = self.wandb.Artifact(f"{self.run.id}-{prefix}-reports", type="evaluation")\n        for p in directory.iterdir():\n            if p.is_file():\n                artifact.add_file(str(p), name=p.name)\n        self.run.log_artifact(artifact)\n\n    def finish(self, failed=False):\n        if self.run:\n            self.run.finish(exit_code=1 if failed else 0)\n\n\ndef save_rng(generator):\n    state = np.random.get_state()\n    return {"python": random.getstate(), "numpy": (state[0], state[1].tolist(), state[2], state[3], state[4]),\n            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n            "loader": generator.get_state()}\n\n\ndef restore_rng(state, generator):\n    random.setstate(state["python"])\n    n = state["numpy"]\n    np.random.set_state((n[0], np.asarray(n[1], dtype=np.uint32), n[2], n[3], n[4]))\n    torch.set_rng_state(state["torch"])\n    if torch.cuda.is_available() and state["cuda"]:\n        torch.cuda.set_rng_state_all(state["cuda"])\n    generator.set_state(state["loader"])\n\n\ndef move_optimizer(optimizer, device):\n    for state in optimizer.state.values():\n        for key, value in state.items():\n            if torch.is_tensor(value):\n                state[key] = value.to(device)\n\n\ndef train_experiment(stage, crops, crop_root, output_root, config, manifest_report, crop_archive_sha256):\n    require(stage in STAGES, "Unknown stage")\n    cfg = copy.deepcopy(config)\n    require(cfg["epochs"] >= 1 and cfg["batch_size"] >= 1 and cfg["num_workers"] >= 0, "Invalid training configuration")\n    require(cfg["image_size"] == 224 and cfg["augmentation"] == DEFAULT_CONFIG["augmentation"], "Update transform code and protocol together")\n    directory = Path(output_root) / stage\n    directory.mkdir(parents=True, exist_ok=True)\n    spec = {"config": cfg, "stage": stage, "classes": CLASSES,\n            "manifest_sha256": manifest_report["manifest_sha256"], "archive_sha256": crop_archive_sha256,\n            "code_sha256": source_hashes()}\n    spec_hash = fingerprint(spec)\n    spec_path = directory / "experiment.json"\n    if spec_path.exists():\n        require(fingerprint(json.loads(spec_path.read_text(encoding="utf-8"))) == spec_hash,\n                "This run folder contains another configuration/data/code version. Use a new output folder.")\n    else:\n        save_json(spec_path, spec)\n    if (directory / "complete.json").exists():\n        print(f"{stage}: training already complete; using saved validation results.")\n        return json.loads((directory / "val/metrics.json").read_text())\n    seed_everything(cfg["seed"])\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    train_rows = crops[crops.split.eq("train")].copy()\n    val_rows = crops[crops.split.eq("val")].copy()\n    generator = torch.Generator().manual_seed(cfg["seed"])\n    train_loader = make_loader(train_rows, crop_root, cfg, True, generator)\n    val_loader = make_loader(val_rows, crop_root, cfg)\n    last_path = directory / "last_checkpoint.pt"\n    model = build_model(stage, pretrained=not last_path.exists()).to(device)\n    lr = cfg["scratch_lr"] if stage == "scratch" else cfg["transfer_lr"]\n    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=lr,\n                                 weight_decay=cfg["weight_decay"])\n    history, best_f1, best_epoch, start_epoch = [], -1.0, 0, 1\n    if last_path.exists():\n        checkpoint = torch.load(last_path, map_location="cpu", weights_only=True)\n        require(checkpoint["spec_hash"] == spec_hash, "Checkpoint/config mismatch")\n        model.load_state_dict(checkpoint["model"])\n        optimizer.load_state_dict(checkpoint["optimizer"])\n        move_optimizer(optimizer, device)\n        history, best_f1 = checkpoint["history"], checkpoint["best_f1"]\n        best_epoch, start_epoch = checkpoint["best_epoch"], checkpoint["epoch"] + 1\n        restore_rng(checkpoint["rng"], generator)\n        print(f"{stage}: resuming after epoch {start_epoch - 1}.")\n    meta_path = directory / "tracking.json"\n    run_id = json.loads(meta_path.read_text())["run_id"] if meta_path.exists() else uuid.uuid4().hex[:12]\n    save_json(meta_path, {"run_id": run_id, "mode": cfg["wandb_mode"]})\n    env = environment()\n    save_json(directory / "environment.json", env)\n    frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)\n    (directory / "requirements.lock.txt").write_text(frozen.stdout, encoding="utf-8")\n    for name in spec["code_sha256"]:\n        shutil.copyfile(Path(__file__).parent / name, directory / name)\n    tracker = Tracker({**cfg, "manifest_sha256": spec["manifest_sha256"], "archive_sha256": crop_archive_sha256,\n                       "code_sha256": spec["code_sha256"], "environment": env}, stage, directory, run_id=run_id)\n    save_json(meta_path, {"run_id": run_id, "mode": cfg["wandb_mode"],\n                         "url": tracker.run.url if tracker.run else None})\n    failed = True\n    try:\n        tracker.summary({"parameters_total": sum(p.numel() for p in model.parameters()),\n                         "parameters_trainable": sum(p.numel() for p in model.parameters() if p.requires_grad)})\n        for epoch in range(start_epoch, cfg["epochs"] + 1):\n            tick = time.monotonic()\n            training_mode(model, stage)\n            loss_sum, ys, probs = 0.0, [], []\n            for x, y in train_loader:\n                x, y = x.to(device), y.to(device)\n                optimizer.zero_grad(set_to_none=True)\n                logits = model(x)\n                loss = nn.functional.cross_entropy(logits, y)\n                require(torch.isfinite(loss).item(), "Non-finite training loss")\n                loss.backward()\n                optimizer.step()\n                loss_sum += loss.item() * len(y)\n                ys.append(y.detach().cpu().numpy())\n                probs.append(logits.detach().softmax(dim=1).cpu().numpy())\n            train_m, _ = classification_metrics(np.concatenate(ys), np.concatenate(probs))\n            y_val, p_val, val_loss = evaluate(model, val_loader, device)\n            val_m, _ = classification_metrics(y_val, p_val)\n            row = {"epoch": epoch, "train/loss": loss_sum / len(train_rows), "val/loss": val_loss,\n                   "train/accuracy": train_m["accuracy"], "train/macro_f1": train_m["macro_f1"],\n                   **flat_metrics(val_m, "val"), "epoch_seconds": time.monotonic() - tick}\n            history.append(row)\n            if val_m["macro_f1"] > best_f1:\n                best_f1, best_epoch = val_m["macro_f1"], epoch\n                atomic_checkpoint(directory / "best_checkpoint.pt",\n                                  {"model": model.state_dict(), "classes": CLASSES, "stage": stage,\n                                   "epoch": epoch, "spec_hash": spec_hash})\n            atomic_checkpoint(last_path, {"model": model.state_dict(), "optimizer": optimizer.state_dict(),\n                                          "epoch": epoch, "best_epoch": best_epoch, "best_f1": best_f1,\n                                          "history": history, "rng": save_rng(generator), "spec_hash": spec_hash})\n            pd.DataFrame(history).to_csv(directory / "learning_curves.csv", index=False)\n            tracker.log(row)\n            print(f"{stage} {epoch}/{cfg[\'epochs\']} | loss {row[\'train/loss\']:.4f} | val F1 {val_m[\'macro_f1\']:.4f}", flush=True)\n        checkpoint = torch.load(directory / "best_checkpoint.pt", map_location="cpu", weights_only=True)\n        model.load_state_dict(checkpoint["model"])\n        y_val, p_val, _ = evaluate(model, val_loader, device)\n        val_metrics = save_evaluation(val_rows, y_val, p_val, crop_root, directory / "val", "val",\n                                      cfg["seed"], cfg["bootstrap_draws"])\n        plot_history(history, directory / "learning_curves.png")\n        tracker.summary({"best_epoch": best_epoch, "selection": "validation macro F1",\n                         **flat_metrics(val_metrics, "best_val")})\n        tracker.reports(directory / "val", "best_val")\n        if tracker.run:\n            tracker.log({"learning_curves": tracker.wandb.Image(str(directory / "learning_curves.png"))})\n        failed = False\n    finally:\n        tracker.finish(failed)\n        del model, optimizer, train_loader, val_loader\n        gc.collect()\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n    save_json(directory / "complete.json", {"spec_hash": spec_hash, "best_epoch": best_epoch,\n                                           "best_val_macro_f1": best_f1})\n    return val_metrics\n\n\ndef final_evaluation(crops, crop_root, output_root, config, manifest_report, crop_archive_sha256, confirm=False):\n    require(confirm is True, "Finish validation review and freeze the three experiments before final test.")\n    root = Path(output_root)\n    specs = {}\n    for stage in STAGES:\n        directory = root / stage\n        require((directory / "complete.json").exists(), f"{stage} training is incomplete")\n        spec = json.loads((directory / "experiment.json").read_text(encoding="utf-8"))\n        require(spec["config"] == config and spec["code_sha256"] == source_hashes(), "Code/config changed after training")\n        require(spec["manifest_sha256"] == manifest_report["manifest_sha256"] and spec["archive_sha256"] == crop_archive_sha256,\n                "Data changed since training")\n        val_m = json.loads((directory / "val/metrics.json").read_text())\n        specs[stage] = {"checkpoint_sha256": sha256(directory / "best_checkpoint.pt"),\n                        "spec_hash": fingerprint(spec), "val_macro_f1": val_m["macro_f1"]}\n    selected = max(STAGES, key=lambda stage: specs[stage]["val_macro_f1"])\n    plan = {"experiments": specs, "selected_by_validation": selected, "selection_metric": "macro_f1"}\n    lock = root / "final_test_plan.json"\n    if lock.exists():\n        require(json.loads(lock.read_text()) == plan, "Final test plan already locked to different checkpoints")\n    else:\n        save_json(lock, plan)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    rows = crops[crops.split.eq("test")].copy()\n    comparison = []\n    for stage in STAGES:\n        directory = root / stage\n        out = directory / "test"\n        if not (out / "complete.json").exists():\n            model = build_model(stage, pretrained=False).to(device)\n            checkpoint = torch.load(directory / "best_checkpoint.pt", map_location="cpu", weights_only=True)\n            require(checkpoint["spec_hash"] == specs[stage]["spec_hash"], "Test checkpoint does not match locked experiment")\n            model.load_state_dict(checkpoint["model"])\n            loader = make_loader(rows, crop_root, config)\n            y, p, _ = evaluate(model, loader, device)\n            m = save_evaluation(rows, y, p, crop_root, out, "test", config["seed"], config["bootstrap_draws"])\n            tracker = Tracker({**config, "final_test_plan": plan}, stage, directory, kind="final_test")\n            try:\n                tracker.summary({**flat_metrics(m, "test"), "best_epoch": checkpoint["epoch"],\n                                 "selected_by_validation": stage == selected})\n                tracker.reports(out, "test")\n                url = tracker.run.url if tracker.run else None\n            except BaseException:\n                tracker.finish(True)\n                raise\n            tracker.finish()\n            save_json(out / "complete.json", {"checkpoint_sha256": specs[stage]["checkpoint_sha256"], "wandb_url": url})\n            del model, loader\n            gc.collect()\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n        require(json.loads((out / "complete.json").read_text())["checkpoint_sha256"] == specs[stage]["checkpoint_sha256"],\n                "Saved test results came from a different checkpoint")\n        m = json.loads((out / "metrics.json").read_text())\n        comparison.append({"model": stage, "selected_by_validation": stage == selected,\n                           "val_macro_f1": specs[stage]["val_macro_f1"],\n                           **{f"test_{k}": v for k, v in m.items() if isinstance(v, (int, float))},\n                           "test_accuracy_ci95_low": m["accuracy_ci95"]["lower"],\n                           "test_accuracy_ci95_high": m["accuracy_ci95"]["upper"]})\n    table = pd.DataFrame(comparison)\n    table.to_csv(root / "comparison.csv", index=False)\n    if config["wandb_mode"] != "disabled" and not (root / "comparison_logged.json").exists():\n        tracker = Tracker(config, "comparison", root, kind="comparison")\n        try:\n            tracker.log({"model_comparison": tracker.wandb.Table(dataframe=table)})\n        except BaseException:\n            tracker.finish(True)\n            raise\n        tracker.finish()\n        save_json(root / "comparison_logged.json", {"plan_hash": fingerprint(plan)})\n    return table\n\n'}

CODE_DIR = Path("/content/road_damage_m2m3_code")
CODE_DIR.mkdir(parents=True, exist_ok=True)
for name, source in MODULE_SOURCES.items():
    (CODE_DIR / name).write_text(source, encoding="utf-8")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from m2m3_data import validate_manifests, verify_pixels, extract_crops, save_json, sha256
from m2m3_train import DEFAULT_CONFIG, train_experiment, final_evaluation
print("Training code ready.")


## 4. Validate the M1 manifests

Check agreement between annotations, crops and splits. The old machine-specific paths in the CSV files are not used to load images.

Do not rerun preprocessing or create new splits. Expected crop counts: train = 12,000; validation = 9,463; test = 11,229.


In [ ]:
import shutil
LOCAL_INPUTS = Path("/content/road_damage_m2m3_inputs")
LOCAL_INPUTS.mkdir(parents=True, exist_ok=True)
for name in required_files[:3]:
    shutil.copyfile(DATA_DIR / name, LOCAL_INPUTS / name)
crops, manifest_report = validate_manifests(LOCAL_INPUTS, strict=True)
EXPECTED_MANIFEST_HASHES = {'annotations.csv': 'c68af856e49181af0ed2133739390b10a6bd25b99daba24061e8862a7451d84d', 'classification_manifest.csv': '1ea96d97860f74ea2ef06d95a31f33d431462a297492d4650e76a04c3badf369', 'split_manifest.csv': '923e6d22b73992eb57b5287dc7c9f45e6554ff555d1dfa4912ba9c1b3c35aa86'}
if manifest_report["manifest_sha256"] != EXPECTED_MANIFEST_HASHES:
    raise ValueError("The CSV files differ from the reviewed snapshot. Review the changes before training.")
save_json(OUTPUT_DIR / "manifest_validation.json", manifest_report)
display(pandas.crosstab(crops["class"], crops["split"])[["train", "val", "test"]])
print("Manifest checks passed.")


## 5. Prepare and verify the images

Copy the ZIP to Colab's local storage and extract it there for faster image loading. Results and checkpoints remain in Drive.

Check that every file can be decoded and has the expected dimensions and color mode. These checks do not establish the visual correctness of every label.

If extraction is interrupted and the cache directory is incomplete, change LOCAL_CROPS to a new directory name. Keep the Drive files intact.


In [ ]:
LOCAL_ZIP = LOCAL_INPUTS / "classification_crops.zip"
if not LOCAL_ZIP.exists() or sha256(LOCAL_ZIP) != sha256(DATA_DIR / "classification_crops.zip"):
    shutil.copyfile(DATA_DIR / "classification_crops.zip", LOCAL_ZIP)
LOCAL_CROPS = Path("/content/rdd2022_crops_cache_v1")
extract_crops(LOCAL_ZIP, LOCAL_CROPS)
pixel_report = verify_pixels(crops, LOCAL_CROPS)
ARCHIVE_SHA256 = sha256(LOCAL_ZIP)
save_json(OUTPUT_DIR / "pixel_validation.json", {**pixel_report, "archive_sha256": ARCHIVE_SHA256})
print(pixel_report)


## 6. Configure W&B and the experiments

Sign in using the prompt inside Colab. Do not store your API key in the notebook or share it in a chat.

WANDB_ENTITY is optional: leave it empty for your default account or enter your W&B team name.

Start with the supplied configuration. If you change it after training has started, select a new OUTPUT_FOLDER.


In [ ]:
import wandb
wandb.login()
CONFIG = dict(DEFAULT_CONFIG)
CONFIG.update(wandb_entity=WANDB_ENTITY.strip() or None,
              wandb_project=WANDB_PROJECT, group=OUTPUT_DIR.name, wandb_mode="online")
if (OUTPUT_DIR / "protocol.json").exists():
    import json
    if json.loads((OUTPUT_DIR / "protocol.json").read_text()) != CONFIG:
        raise ValueError("The configuration differs from a previous experiment. Choose a new OUTPUT_FOLDER.")
save_json(OUTPUT_DIR / "protocol.json", CONFIG)
display(pandas.Series(CONFIG, name="value").to_frame())


## 7. Check one data-loading batch

This operational check uses only the training and validation splits. Training metrics are measured during optimization with image augmentation, so they differ from a fixed evaluation of the training set.


In [ ]:
from m2m3_train import make_loader
for split in ["train", "val"]:
    loader = make_loader(crops[crops.split.eq(split)], LOCAL_CROPS, CONFIG, training=(split == "train"))
    images, labels = next(iter(loader))
    assert images.shape[1:] == (3, 224, 224)
    assert torch.isfinite(images).all() and labels.min() >= 0 and labels.max() < 4
    print(split, tuple(images.shape), sorted(labels.unique().tolist()))
del loader, images, labels


## 8. M2 — Train from scratch

Run this cell and wait for it to finish. If the runtime disconnects, rerun the setup cells and then this cell using the same output folder and configuration. Training resumes from the last saved epoch.

Use best_checkpoint.pt for evaluation and last_checkpoint.pt for resuming training. The Norway test set is not evaluated here.


In [ ]:
scratch_validation = train_experiment("scratch", crops, LOCAL_CROPS, OUTPUT_DIR, CONFIG,
                                       manifest_report, ARCHIVE_SHA256)
display(pandas.DataFrame(scratch_validation["per_class"]).T)


## 9. M3 — Train the frozen ResNet18 model

Train only the classification head. Backbone parameters and BatchNorm statistics remain fixed.


In [ ]:
frozen_validation = train_experiment("frozen", crops, LOCAL_CROPS, OUTPUT_DIR, CONFIG,
                                      manifest_report, ARCHIVE_SHA256)
display(pandas.DataFrame(frozen_validation["per_class"]).T)


## 10. M3 — Fine-tune all ResNet18 layers

This is an independent experiment that starts from the same ImageNet weights and trains all layers. It does not resume the frozen-head experiment.


In [ ]:
finetune_validation = train_experiment("finetune", crops, LOCAL_CROPS, OUTPUT_DIR, CONFIG,
                                        manifest_report, ARCHIVE_SHA256)
display(pandas.DataFrame(finetune_validation["per_class"]).T)


## 11. Review validation results and analyze errors

Open the printed W&B links. Review the following files in each experiment's val directory:

- per_class_metrics.csv and confusion_matrix.png.
- reliability.png and metrics.json: lower ECE is better. Calibration is measured on raw softmax probabilities, without post-hoc correction.
- error_gallery.png shows the highest-confidence errors. error_review.csv combines these with a random error sample for manual review.
- data_improvement_plan.md is a provisional plan based on class-level misses and confusion pairs. Record observed causes after inspecting the images; the generated plan is not a completed visual error analysis.

Select the model using validation macro F1. Do not rely on accuracy alone because the validation and test classes are imbalanced.


In [ ]:
import json
from IPython.display import display, Image as DisplayImage
validation_rows = []
for stage in ["scratch", "frozen", "finetune"]:
    metrics = json.loads((OUTPUT_DIR / stage / "val/metrics.json").read_text())
    tracking = json.loads((OUTPUT_DIR / stage / "tracking.json").read_text())
    validation_rows.append({"model":stage, **{k:v for k,v in metrics.items() if isinstance(v,(float,int))}})
    print(stage, tracking.get("url"))
validation_comparison = pandas.DataFrame(validation_rows).sort_values("macro_f1", ascending=False)
display(validation_comparison)
SELECTED_STAGE = validation_comparison.iloc[0]["model"]
print("Selected using validation:", SELECTED_STAGE)
for name in ["confusion_matrix.png", "reliability.png", "error_gallery.png"]:
    display(DisplayImage(filename=str(OUTPUT_DIR / SELECTED_STAGE / "val" / name)))
print((OUTPUT_DIR / SELECTED_STAGE / "val/data_improvement_plan.md").read_text())


## 12. Run final test evaluation after validation review

Keep the option below set to False during development. Set it to True only after fixing the three experiments and selecting the model using validation results.

The notebook records a plan tied to specific checkpoint files, then evaluates all three models on the same Norway test set. Do not tune models using test results and subsequently describe that same test set as untouched.

The accuracy confidence interval resamples **parent images**, keeping correlated crops together. It does not cover variation across training seeds or generalization to every country.


In [ ]:
# @title Freeze the experiments and run final test evaluation
FINAL_TEST_READY = False # @param {type:"boolean"}
if FINAL_TEST_READY:
    comparison = final_evaluation(crops, LOCAL_CROPS, OUTPUT_DIR, CONFIG,
                                  manifest_report, ARCHIVE_SHA256, confirm=True)
    display(comparison)
else:
    print("Final test evaluation is disabled. Complete validation review, then set FINAL_TEST_READY to True.")


## 13. Submission files and limitations

After training and final evaluation, use the actual outputs from Drive and W&B:

1. The W&B project link and comparison of the three experiments.
2. comparison.csv, learning curves and best_checkpoint.pt for each experiment.
3. Per-class metrics, confusion matrices, reliability diagrams and metrics.json, with interpretation.
4. A manually reviewed error sample and a concrete improvement plan supported by counts and examples.
5. environment.json, requirements.lock.txt, experiment.json and manifest_validation.json for reproducibility.

**Unresolved M1 audit issue:** Some label-audit decisions appear to be associated with row numbers from another sample. Match decisions to image_id/object_index with your teammate before reporting the label-error rate. This notebook does not use those decisions to modify labels.

**Experiment limitations:** Country separation evaluates transfer across data sources. Norway does not represent all other countries, and one fixed seed does not measure training variability. The M1 crops have already been resized to 224 × 224 squares; this notebook cannot recover details lost during preprocessing.

Implementation references:

- [PyTorch ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)
- [PyTorch BatchNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html)
- [W&B logging](https://docs.wandb.ai/models/track/log)
- [Colab FAQ](https://research.google.com/colaboratory/faq.html)
